# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/betmutema/ml-engineering-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import os, getpass, duckdb, pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"
ANCHOR = "DATE '2026-03-31'"

features = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
        SUM(CASE WHEN f.report_date >  {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
        AVG(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_avg_position END)        AS pos_prev30
    FROM {fact} f
    WHERE f.report_date BETWEEN {ANCHOR} - INTERVAL 60 DAY AND {ANCHOR}
    GROUP BY 1,2
    HAVING imp_prev30 >= 100
""").df()
features['ctr_prev30'] = features['clk_prev30'] / features['imp_prev30']
features = features.merge(
    con.sql(f"""SELECT content_hash_id, DATE_DIFF('day', content_created_date, {ANCHOR}) AS content_age_days
                FROM {dim_content}""").df(),
    on='content_hash_id', how='left'
)
features['is_declining'] = (features['imp_last30'] < 0.8 * features['imp_prev30']).astype(int)
features = features.dropna(subset=['pos_prev30', 'content_age_days']).reset_index(drop=True)
print(f"{len(features):,} content items")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

82,549 content items


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
FEATURE_COLS = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30', 'content_age_days']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(features, groups=features['client_hash_id']))
train, test = features.iloc[train_idx].copy(), features.iloc[test_idx].copy()

print(f"Train: {len(train):,} rows, {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test):,} rows, {test['client_hash_id'].nunique()} clients")
print(f"Client overlap between train/test: {len(set(train.client_hash_id) & set(test.client_hash_id))}")

Train: 56,191 rows, 27 clients
Test:  26,358 rows, 10 clients
Client overlap between train/test: 0


**is_declining** is an observed yes/no label, so per the toolkit this starts with Logistic Regression (readable) then Random Forest (stronger). This is a ranking task in practice (**precision@50**), so both models are evaluated on their predicted probability, not their hard 0/1 label.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X_train, y_train = train[FEATURE_COLS], train['is_declining']
X_test, y_test = test[FEATURE_COLS], test['is_declining']

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

stale = (test['content_age_days'] >= 365).astype(int)
visible = (test['imp_prev30'] >= 250).astype(int)
baseline_scores = stale * visible * test['imp_prev30']

comparison = pd.DataFrame({
    'method': ['Base rate', 'Baseline rule (ML-07)', 'Logistic Regression', 'Random Forest'],
    'precision_at_50': [
        y_test.mean(),
        precision_at_k(baseline_scores, y_test, 50),
        precision_at_k(logreg_scores, y_test, 50),
        precision_at_k(rf_scores, y_test, 50),
    ]
}).round(3)
print(comparison)

                  method  precision_at_50
0              Base rate            0.279
1  Baseline rule (ML-07)            0.100
2    Logistic Regression            0.100
3          Random Forest            0.200


Split by client, not content item, so evaluation reflects performance on genuinely unseen clients, not just unseen pages from clients the model already learned from, directly motivated by the top-20 client concentration seen in ML-07

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=False)
print("Permutation importance (Random Forest):")
print(importance.round(4))

test['rf_score'] = rf_scores
top_wrong = test.sort_values('rf_score', ascending=False).head(50)
false_positives = top_wrong[top_wrong['is_declining'] == 0]
print(f"\n{len(false_positives)} of the top 50 by model score are NOT actually declining")
false_positives[['client_hash_id','content_age_days','imp_prev30','imp_last30']].head(5)

Permutation importance (Random Forest):
ctr_prev30         -0.0006
pos_prev30         -0.0029
content_age_days   -0.0037
imp_prev30         -0.0058
clk_prev30         -0.0079
dtype: float64

40 of the top 50 by model score are NOT actually declining


,client_hash_id,content_age_days,imp_prev30,imp_last30
2647,client_62f4a7e64f5e0096,248,123.0,189.0
2104,client_62f4a7e64f5e0096,265,129.0,220.0
1995,client_62f4a7e64f5e0096,265,122.0,194.0
43360,client_62f4a7e64f5e0096,265,131.0,160.0
2522,client_62f4a7e64f5e0096,249,134.0,120.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.